In [1]:
!pip install "https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3%2Bcu12torch2.9cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.8/253.8 MB 9.5 MB/s eta 0:00:0000:0100:01


In [14]:
!rm -rf "/content/SimpleVLLM"

In [15]:
!git clone https://github.com/Ajax0564/SimpleVLLM.git

Cloning into 'SimpleVLLM'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (60/60), done.
remote: Total 98 (delta 39), reused 83 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 121.43 KiB | 24.29 MiB/s, done.
Resolving deltas: 100% (39/39), done.


In [16]:
import os
import sys

# Point directly to the 'src' directory
src_path = os.path.abspath("/content/SimpleVLLM/src")
if src_path not in sys.path:
    sys.path.append(src_path)

print("Successfully added to sys.path:", src_path)

Successfully added to sys.path: /content/SimpleVLLM/src


In [17]:
import sys

# Purge all simplevllm modules from Python's cache
for module in list(sys.modules.keys()):
    if module.startswith("simplevllm"):
        del sys.modules[module]

# Re-import core engine and model functions
from simplevllm.engine import ContinuousBatchEngine, PagedKVManager, SequenceState
from simplevllm.models import Qwen3Config, get_qwen3_model, get_qwen3_tokenizer

print("✓ Successfully imported Qwen3Config and SimpleVLLM modules!")
print("Config max_batch_size:", Qwen3Config["max_batch_size"])

✓ Successfully imported Qwen3Config and SimpleVLLM modules!
Config max_batch_size: 8


In [18]:
# Load model and tokenizer
tokenizer = get_qwen3_tokenizer()
model = get_qwen3_model()

print("✓ Model and Tokenizer initialized on device successfully!")

✓ Model and Tokenizer initialized on device successfully!


In [19]:
import torch
torch.manual_seed(123)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)

In [20]:
from simplevllm.models import Qwen3Config
mgr = PagedKVManager(Qwen3Config, max_blocks=32, device=device)

In [21]:
prompt = "Give me a short introduction to large language models."

input_token_ids1 = tokenizer.encode(prompt)

In [22]:
prompt = "Give me a short introduction to AI"

input_token_ids3 = tokenizer.encode(prompt)

In [23]:
prompt = "Give me a short introduction to Embeddings in nlp"

input_token_ids2 = tokenizer.encode(prompt)

In [24]:
batched_engine = ContinuousBatchEngine(model, mgr, Qwen3Config)
print("BatchedEngine ready")

BatchedEngine ready


In [25]:
my_prompts = [
    input_token_ids1,
    input_token_ids2,
    input_token_ids3
]
batched_engine.reset()
for prompt in my_prompts:
    batched_engine.add_sequence(prompt,max_gen_len = 512)


while batched_engine.waiting_room or batched_engine.active:
    finished = batched_engine.step()
     # Print results as they come in
    for sid, tokens in finished.items():
        print(f"Slot {sid} finished. Total length: {len(tokens)}")
        print(tokenizer.decode(tokens))
        print("-" * 30)

Slot 2 finished. Total length: 180
<|im_start|>user
Give me a short introduction to AI<|im_end|>
<|im_start|>assistant
<think>
Okay, the user wants a short introduction to AI. Let me start by defining what AI is. I should mention it's a technology that helps machines think and learn. Maybe include some key points like machine learning, natural language processing, and applications. Need to keep it concise but informative. Also, make sure to highlight the benefits and the importance of AI in modern life. Avoid technical jargon to keep it accessible. Let me check if I'm covering all important aspects without being too long. Alright, that should work.
</think>

AI, or artificial intelligence, is a technology that enables machines to think and learn from data. It uses algorithms and machine learning to analyze information and make decisions automatically. AI has applications in various fields, from healthcare to finance, and is transforming how we interact with the world.<|im_end|>
-------

## Launch the multi-turn chat app

The next cells reuse the GPU model, tokenizer, and batched engine already created above. Run the setup cell once, then run the server cell and open the displayed URL.

In [32]:
%pip install -q fastapi "uvicorn[standard]"

from simplevllm.chat_app import ChatRuntime, create_app

# Reuse the model and engine already loaded on the notebook GPU.
notebook_runtime = ChatRuntime(
    model=model,
    tokenizer=tokenizer,
    engine=batched_engine,
)
chat_app = create_app(notebook_runtime)
print("Chat app is configured with the existing GPU model")

Chat app is configured with the existing GPU model


In [55]:
import os
import socket
import threading
import time
import traceback

import uvicorn
from IPython.display import HTML, display

# Pick an available notebook port so rerunning this cell is safe.
def find_free_port(start=8000, end=8010):
    for candidate in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
            try:
                probe.bind(("127.0.0.1", candidate))
                return candidate
            except OSError:
                continue
    raise RuntimeError("No free port found between 8000 and 8010")

# Stop a previous server before starting a new one.
if "chat_server" in globals() and "chat_thread" in globals() and chat_thread.is_alive():
    chat_server.should_exit = True
    chat_thread.join(timeout=5)

chat_port = find_free_port()
server_error = None
server_config = uvicorn.Config(
    chat_app,
    host="0.0.0.0",
    port=chat_port,
    log_level="warning",
)
chat_server = uvicorn.Server(server_config)


def run_chat_server():
    global server_error
    try:
        chat_server.run()
    except Exception:
        server_error = traceback.format_exc()


chat_thread = threading.Thread(target=run_chat_server, daemon=True)
chat_thread.start()

for _ in range(50):
    if server_error:
        raise RuntimeError(server_error)
    if not chat_thread.is_alive():
        raise RuntimeError("Chat server stopped before it began listening")
    try:
        with socket.create_connection(("127.0.0.1", chat_port), timeout=0.2):
            break
    except OSError:
        time.sleep(0.1)
else:
    if server_error:
        raise RuntimeError(server_error)
    raise RuntimeError(f"Chat server did not start on port {chat_port}")

if os.getenv("COLAB_RELEASE_TAG"):
    from google.colab import output

    print(f"Chat server is running on the remote Colab kernel at port {chat_port}.")
    print("Open this notebook in the Google Colab browser to display the embedded chat window.")
    output.serve_kernel_port_as_iframe(chat_port)
else:
    chat_url = f"http://127.0.0.1:{chat_port}"
    print(f"Chat app is running at {chat_url}")
    display(HTML(f'<a href="{chat_url}" target="_blank">Open SimpleVLLM Chat</a>'))

Chat server is running on the remote Colab kernel at port 8001.
Open this notebook in the Google Colab browser to display the embedded chat window.


<IPython.core.display.Javascript object>

In [43]:
# Run this cell when you want to stop the current notebook server.
if "chat_server" in globals():
    chat_server.should_exit = True
if "chat_thread" in globals():
    chat_thread.join(timeout=5)
print("Chat server stopped")

Chat server stopped
